# Evaluation: Simulation-Pretrained Transformer vs RF Baseline

## Metrics
- ROC-AUC with bootstrap 95% CI (200 resamples, using split.bootstrap_roc_auc)
- PR-AUC (Average Precision)
- Best F1 threshold (selected on validation set)
- Confusion matrix at optimal threshold

## Comparisons
1. Simulation-pretrained + adversarial fine-tuned Transformer
2. RF baseline (16 physical features, AUC = 0.7994)
3. Ablation: Transformer trained from scratch on real data only (no pretraining)
4. Ablation: Simulation-pretrained WITHOUT domain adaptation (no adversarial)

In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "0"

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
from torch.utils.data import DataLoader, Dataset

seed = 42
random.seed(seed); np.random.seed(seed)
torch.manual_seed(seed); torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

observations = pd.read_pickle('/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl')
from split import ensure_split, bootstrap_roc_auc
train_stars, val_stars, test_stars = ensure_split(observations)

NUM_FREQS = 8
PERIODS = np.logspace(np.log10(1.0), np.log10(7300.0), NUM_FREQS)
FREQS = 2.0 * np.pi / PERIODS

pos_stars = sorted(set(observations[observations['has_exoplanets'] == 1]['star_name']))
neg_stars = sorted(set(observations[observations['has_exoplanets'] == 0]['star_name']))

def star_to_features_real(star_name):
    star_obs = observations[observations['star_name'] == star_name].sort_values('bjd')
    ref_bjd = star_obs['bjd'].iloc[0]
    features = []
    for _, row in star_obs.iterrows():
        dt = row['bjd'] - ref_bjd
        pos_enc = []
        for f in FREQS: pos_enc.append(np.sin(f*dt)); pos_enc.append(np.cos(f*dt))
        features.append([row['rv_centered'], row['rv_err'], row['exposure_time'],
                        row['RHKp'], row['Halpha']] + pos_enc)
    return np.array(features, dtype=np.float32)

test_pos = [s for s in test_stars if s in set(pos_stars)]
test_neg = [s for s in test_stars if s in set(neg_stars)]
val_pos = [s for s in val_stars if s in set(pos_stars)]
val_neg = [s for s in val_stars if s in set(neg_stars)]

sim_mean = np.load('/kaggle/working/sim_norm_mean.npy')
sim_std = np.load('/kaggle/working/sim_norm_std.npy')

test_data = [(star_to_features_real(s), 1) for s in test_pos] + [(star_to_features_real(s), 0) for s in test_neg]
test_data = [((seq - sim_mean) / sim_std, l) for seq, l in test_data]
val_data = [(star_to_features_real(s), 1) for s in val_pos] + [(star_to_features_real(s), 0) for s in val_neg]
val_data = [((seq - sim_mean) / sim_std, l) for seq, l in val_data]

from sim_dataset import collate_stars

class StarDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx][0], dtype=torch.float32), torch.tensor(self.data[idx][1], dtype=torch.float32)

test_loader = DataLoader(StarDataset(test_data), batch_size=32, shuffle=False, collate_fn=collate_stars)
val_loader = DataLoader(StarDataset(val_data), batch_size=32, shuffle=False, collate_fn=collate_stars)

In [ ]:
# Load best fine-tuned model and evaluate on test set
from transformer_model import AttentionPool

class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim=48, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3), nn.Linear(hidden_dim, 1))
    def forward(self, z): return self.net(z).squeeze(-1)

class ExoplanetTransformerWithDomain(nn.Module):
    def __init__(self, feat_dim=21, d_model=48, nhead=4, num_layers=1, dim_feedforward=96, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = AttentionPool(d_model)
        self.classifier = nn.Sequential(nn.Linear(d_model, 16), nn.ReLU(), nn.Dropout(dropout), nn.Linear(16, 1))
        self.domain_disc = DomainDiscriminator(input_dim=d_model)
    def encode(self, x, mask):
        x = self.input_proj(x)
        x = self.transformer(x, src_key_padding_mask=~mask.bool())
        return self.pool(x, mask)
    def forward(self, x, mask):
        z = self.encode(x, mask)
        return self.classifier(z).squeeze(-1)

model = ExoplanetTransformerWithDomain().to(device)
model.load_state_dict(torch.load('/kaggle/working/finetuned_adversarial.pth'))
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for padded, mask, labels in test_loader:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        logits = model(padded, mask)
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(labels.numpy())

roc_auc, ci_lo, ci_hi = bootstrap_roc_auc(all_labels, all_probs, n_resamples=200, seed=42)
pr_auc = average_precision_score(all_labels, all_probs)

val_probs, val_labels = [], []
with torch.no_grad():
    for padded, mask, labels in val_loader:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        logits = model(padded, mask)
        val_probs.extend(torch.sigmoid(logits).cpu().numpy())
        val_labels.extend(labels.numpy())

prec_v, rec_v, thr_v = precision_recall_curve(val_labels, val_probs)
f1_v = 2 * prec_v * rec_v / (prec_v + rec_v + 1e-8)
best_idx = np.argmax(f1_v)
best_thresh = thr_v[best_idx] if best_idx < len(thr_v) else 0.5

preds = (np.array(all_probs) >= best_thresh).astype(int)
cm = confusion_matrix(all_labels, preds)

print("=" * 60)
print("Simulation-Pretrained + Adversarial Transformer")
print("=" * 60)
print(f"ROC-AUC: {roc_auc:.4f} [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"PR-AUC:  {pr_auc:.4f}")
print(f"Best F1 threshold (from val): {best_thresh:.4f}")
prec_test = cm[1,1] / (cm[1,1] + cm[0,1]) if cm[1,1] + cm[0,1] > 0 else 0
rec_test = cm[1,1] / (cm[1,1] + cm[1,0]) if cm[1,1] + cm[1,0] > 0 else 0
f1_test = 2 * prec_test * rec_test / (prec_test + rec_test) if prec_test + rec_test > 0 else 0
print(f"Test F1: {f1_test:.4f} (Precision: {prec_test:.4f}, Recall: {rec_test:.4f})")
print("Confusion Matrix:")
print(cm)

In [ ]:
# Ablation 1: Transformer trained from scratch on real data only (no pretraining)
# This should reproduce ~0.6353 AUC from state.md
from transformer_model import ExoplanetTransformer

model_scratch = ExoplanetTransformer(feat_dim=21, d_model=48, nhead=4, num_layers=1, dim_feedforward=96, dropout=0.3).to(device)

train_pos = [s for s in train_stars if s in set(pos_stars)]
train_neg = [s for s in train_stars if s in set(neg_stars)]
train_data_scratch = [(star_to_features_real(s), 1) for s in train_pos] + [(star_to_features_real(s), 0) for s in train_neg]
train_data_scratch = [((seq - sim_mean) / sim_std, l) for seq, l in train_data_scratch]

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([math.sqrt(len(train_neg)/len(train_pos))]).to(device))
optimizer = Adam(model_scratch.parameters(), lr=1e-3, weight_decay=5e-3)

train_loader_scratch = DataLoader(StarDataset(train_data_scratch), batch_size=32, shuffle=True, collate_fn=collate_stars, pin_memory=True)

for epoch in range(50):
    model_scratch.train()
    for padded, mask, labels in train_loader_scratch:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_scratch(padded, mask), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_scratch.parameters(), max_norm=1.0)
        optimizer.step()

model_scratch.eval()
scratch_probs, scratch_labels = [], []
with torch.no_grad():
    for padded, mask, labels in test_loader:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        scratch_probs.extend(torch.sigmoid(model_scratch(padded, mask)).cpu().numpy())
        scratch_labels.extend(labels.numpy())

scratch_auc, s_lo, s_hi = bootstrap_roc_auc(scratch_labels, scratch_probs, n_resamples=200, seed=42)
print(f"From-scratch Transformer: ROC-AUC = {scratch_auc:.4f} [{s_lo:.4f}, {s_hi:.4f}]")

In [ ]:
# Ablation 2: Simulation-pretrained WITHOUT domain adaptation
# Fine-tune pretrained encoder with lambda_domain = 0 (no adversarial loss)
model_noadv = ExoplanetTransformerWithDomain().to(device)
pretrained = torch.load('/kaggle/working/pretrained_stage2.pth')
pretrained_keys = {k: v for k, v in pretrained.items() if not k.startswith('classifier')}
model_noadv.load_state_dict(pretrained_keys, strict=False)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([math.sqrt(len(train_neg)/len(train_pos))]).to(device))
optimizer_noadv = Adam(list(model_noadv.input_proj.parameters()) + list(model_noadv.transformer.parameters()) + list(model_noadv.pool.parameters()) + list(model_noadv.classifier.parameters()), lr=1e-4, weight_decay=5e-3)

for epoch in range(50):
    model_noadv.train()
    for padded, mask, labels in train_loader_scratch:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        optimizer_noadv.zero_grad()
        logits, _, _ = model_noadv(padded, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_noadv.parameters(), max_norm=1.0)
        optimizer_noadv.step()

model_noadv.eval()
noadv_probs, noadv_labels = [], []
with torch.no_grad():
    for padded, mask, labels in test_loader:
        padded, mask, labels = padded.to(device), mask.to(device), labels.to(device)
        logits, _, _ = model_noadv(padded, mask)
        noadv_probs.extend(torch.sigmoid(logits).cpu().numpy())
        noadv_labels.extend(labels.numpy())

noadv_auc, n_lo, n_hi = bootstrap_roc_auc(noadv_labels, noadv_probs, n_resamples=200, seed=42)
print(f"Sim-pretrained (no domain adapt): ROC-AUC = {noadv_auc:.4f} [{n_lo:.4f}, {n_hi:.4f}]")

In [ ]:
# Final comparison table
import pandas as pd

comparison = pd.DataFrame([
    {'Model': 'RF (16 physical features)', 'ROC-AUC': 0.7994, '95% CI': '[TBD]', 'PR-AUC': 0.4658, 'Best F1': 0.5221, 'Input': 'per-star aggregates'},
    {'Model': 'Transformer (from scratch, real only)', 'ROC-AUC': scratch_auc, '95% CI': f'[{s_lo:.4f}, {s_hi:.4f}]', 'PR-AUC': float('nan'), 'Best F1': float('nan'), 'Input': 'raw sequences'},
    {'Model': 'Sim-Pretrained Transformer (no adapt)', 'ROC-AUC': noadv_auc, '95% CI': f'[{n_lo:.4f}, {n_hi:.4f}]', 'PR-AUC': average_precision_score(noadv_labels, noadv_probs), 'Best F1': float('nan'), 'Input': 'raw sequences'},
    {'Model': 'Sim-Pretrained + Adversarial (this work)', 'ROC-AUC': roc_auc, '95% CI': f'[{ci_lo:.4f}, {ci_hi:.4f}]', 'PR-AUC': pr_auc, 'Best F1': f1_test, 'Input': 'raw sequences'},
])

print(comparison.to_string(index=False, float_format=lambda x: f'{x:.4f}' if not pd.isna(x) else '—'))